# 261 Mini-Project 2 — Semantic Segmentation on Pascal VOC 2007

This notebook runs the full pipeline:
1. **Setup** — mount Drive, clone repo, install dependencies
2. **Download** — VOC dataset + SAM2 weights (saved to Drive, only once)
3. **Training** — U-Net, DeepLabV3+, SAM2, DINOv2
4. **Inference** — run each model on val samples
5. **Evaluation** — metrics, confusion matrix, ablation studies, visualization, cross-model comparison

---
## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clone repo + set paths

In [ ]:
import os

# Clone (skip if already exists)
if not os.path.exists('/content/261-mini2'):
    !git clone https://github.com/gracee-chen/261-mini2.git /content/261-mini2
else:
    !cd /content/261-mini2 && git pull

%cd /content/261-mini2

In [ ]:
# ---------- All persistent paths (edit if needed) ----------
DRIVE_DATA = '/content/drive/MyDrive/261-data'
DRIVE_CKPT = '/content/drive/MyDrive/261-checkpoints'
DRIVE_RESULTS = '/content/drive/MyDrive/261-results'

VOC_ROOT = f'{DRIVE_DATA}/VOCtrainval_06-Nov-2007'
SAM2_CKPT = f'{DRIVE_DATA}/sam2.1_hiera_base_plus.pt'

# Create Drive directories
for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)

# Symlink checkpoints/ and results/ to Drive for persistence
!ln -sfn {DRIVE_CKPT} checkpoints
!ln -sfn {DRIVE_RESULTS} results

# Create sub-dirs for all models
for m in ['unet', 'deeplabv3plus', 'sam2', 'dinov2', 'dinov2_ft']:
    os.makedirs(f'{DRIVE_CKPT}/{m}', exist_ok=True)

# Set env vars so ! commands can use them
%env VOC_ROOT={VOC_ROOT}
%env SAM2_CKPT={SAM2_CKPT}

print('Done. Checkpoints and results will persist on Drive.')

## 2. Install dependencies

In [ ]:
!pip install -q segmentation-models-pytorch timm transformers scipy matplotlib
!pip install -q git+https://github.com/facebookresearch/sam2.git

## 3. Download dataset + SAM2 weights to Drive (run once)

After the first run these files live on Drive — skip this section on future sessions.

In [ ]:
import os

voc_check = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2007', 'JPEGImages')

if os.path.isdir(voc_check):
    print(f'VOC dataset already exists at {VOC_ROOT}, skipping download.')
else:
    # --- Kaggle credentials ---
    # Option A: Upload kaggle.json from your computer
    from google.colab import files
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    # Option B: kaggle.json already on Drive (uncomment below, comment out Option A)
    # !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    # --- Download ---
    !pip install -q kaggle
    !kaggle datasets download -d zaraks/pascal-voc-2007 -p {DRIVE_DATA} --unzip

    # --- Verify ---
    if os.path.isdir(voc_check):
        print(f'Done. VOC 2007 saved to {VOC_ROOT}')
    else:
        print(f'ERROR: Expected {voc_check} not found.')
        print('The zip structure may differ. Checking what was downloaded:')
        !ls -la {DRIVE_DATA}/

In [ ]:
# ----- SAM2 weights -----
if not os.path.isfile(SAM2_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt \
        -O {SAM2_CKPT}
    print('SAM2 weights downloaded to Drive.')
else:
    print(f'SAM2 weights already at {SAM2_CKPT}')

---
# TRAINING

All checkpoints save to Drive automatically via the symlink.

## 4a. Train U-Net (ResNet-50, ~50 epochs)

In [ ]:
!python train/train_unet.py \
    --voc-root $VOC_ROOT \
    --epochs 50 \
    --checkpoint-dir checkpoints/unet

## 4b. Train DeepLabV3+ (ResNet-50, ~50 epochs)

In [ ]:
!python train/train_deeplabv3plus.py \
    --voc-root $VOC_ROOT \
    --epochs 50 \
    --checkpoint-dir checkpoints/deeplabv3plus

## 4c. Train SAM2 (encoder frozen, ~30 epochs)

In [ ]:
!python train/train_sam2.py \
    --voc-root $VOC_ROOT \
    --sam2-ckpt $SAM2_CKPT \
    --epochs 30 \
    --checkpoint-dir checkpoints/sam2

## 4d. Train DINOv2 — Phase 1 (head only, ~30 epochs)

In [ ]:
!python train/train_dinov2.py \
    --voc-root $VOC_ROOT \
    --epochs 30 \
    --lr 1e-3 \
    --checkpoint-dir checkpoints/dinov2

## 4e. Train DINOv2 — Phase 2 (full fine-tune, ~20 epochs)

In [ ]:
!python train/train_dinov2.py \
    --voc-root $VOC_ROOT \
    --unfreeze-backbone \
    --resume checkpoints/dinov2/best.pth \
    --epochs 20 \
    --lr 1e-5 \
    --checkpoint-dir checkpoints/dinov2_ft

---
# INFERENCE

Run each model on 8 random val samples.

In [ ]:
!python inference/infer_unet.py \
    --checkpoint checkpoints/unet/best.pth \
    --voc-root $VOC_ROOT --num-samples 8 \
    --output-dir results/inference/unet

In [ ]:
!python inference/infer_deeplabv3plus.py \
    --checkpoint checkpoints/deeplabv3plus/best.pth \
    --voc-root $VOC_ROOT --num-samples 8 \
    --output-dir results/inference/deeplabv3plus

In [ ]:
!python inference/infer_sam2.py \
    --checkpoint checkpoints/sam2/best.pth \
    --sam2-ckpt $SAM2_CKPT \
    --voc-root $VOC_ROOT --num-samples 8 \
    --output-dir results/inference/sam2

In [ ]:
!python inference/infer_dinov2.py \
    --checkpoint checkpoints/dinov2_ft/best.pth \
    --voc-root $VOC_ROOT --num-samples 8 \
    --output-dir results/inference/dinov2

---
# EVALUATION

## 6a. Metrics — mIoU, mDice, HD95, Pixel Accuracy (all 4 models)

In [ ]:
!python evaluation/metrics/compute_metrics.py \
    --model-type unet deeplabv3plus sam2 dinov2 \
    --checkpoint checkpoints/unet/best.pth \
                 checkpoints/deeplabv3plus/best.pth \
                 checkpoints/sam2/best.pth \
                 checkpoints/dinov2_ft/best.pth \
    --voc-root  $VOC_ROOT \
    --sam2-ckpt $SAM2_CKPT \
    --output-dir results/metrics

## 6b. Confusion Matrix (U-Net)

In [ ]:
!python evaluation/metrics/confusion_matrix.py \
    --model-type unet \
    --checkpoint checkpoints/unet/best.pth \
    --voc-root $VOC_ROOT \
    --output-dir results/metrics

---
## 7. Ablation Studies (5 experiments, all based on U-Net)

### 7a. Backbone — ResNet-18 vs ResNet-50

In [ ]:
!python evaluation/ablation/ablation_backbone.py \
    --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation

### 7b. Augmentation — None vs Horizontal Flip

In [ ]:
!python evaluation/ablation/ablation_augmentation.py \
    --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation

### 7c. Loss Function — CE vs Dice vs CE+Dice

In [ ]:
!python evaluation/ablation/ablation_loss.py \
    --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation

### 7d. Pretrain — Scratch vs ImageNet

In [ ]:
!python evaluation/ablation/ablation_pretrain.py \
    --voc-root $VOC_ROOT --epochs 25 --output-dir results/ablation

### 7e. Input Resolution — 128 vs 256 vs 384

In [ ]:
!python evaluation/ablation/ablation_resolution.py \
    --voc-root $VOC_ROOT --epochs 25 --batch-size 4 --output-dir results/ablation

---
## 8. Visualization

### 8a. Mosaic — multi-model side-by-side

In [ ]:
!python evaluation/visualization/visualize_mosaic.py \
    --voc-root $VOC_ROOT \
    --models unet:checkpoints/unet/best.pth \
             deeplabv3plus:checkpoints/deeplabv3plus/best.pth \
             dinov2:checkpoints/dinov2_ft/best.pth \
    --num-images 6 \
    --output results/visualization/mosaic.png

### 8b. Top-3 / Worst-3 comparison

In [ ]:
# Ranked by overall mIoU
!python evaluation/visualization/visualize_comparison.py \
    --voc-root $VOC_ROOT \
    --model-type unet deeplabv3plus \
    --checkpoint checkpoints/unet/best.pth \
                 checkpoints/deeplabv3plus/best.pth \
    --metric miou \
    --output-dir results/visualization

# Ranked by person-class IoU
!python evaluation/visualization/visualize_comparison.py \
    --voc-root $VOC_ROOT \
    --model-type unet deeplabv3plus \
    --checkpoint checkpoints/unet/best.pth \
                 checkpoints/deeplabv3plus/best.pth \
    --metric person \
    --output-dir results/visualization

---
## 9. Cross-Model Comparison (summary table + charts)

In [ ]:
!python evaluation/compare.py \
    --models unet deeplabv3plus sam2 dinov2 \
    --checkpoint-dirs checkpoints/unet \
                      checkpoints/deeplabv3plus \
                      checkpoints/sam2 \
                      checkpoints/dinov2_ft \
    --metrics-dir results/metrics \
    --output-dir results/compare

---
## 10. View all output files

In [ ]:
!find results/ -type f | sort

## 11. Display key result images

In [ ]:
from IPython.display import Image, display
import glob

images = sorted(glob.glob('results/**/*.png', recursive=True))
for path in images:
    print(f'\n--- {path} ---')
    display(Image(filename=path, width=800))